In [0]:
from pyspark.sql import functions as F

s = spark.table("chat_bot.silver.sales")
p = spark.table("chat_bot.silver.products")
c = spark.table("chat_bot.silver.customers")

encriched=(s.join(p,"product_id")).join(c,'customer_id')\
          .select(
              
              "sale_id", "sale_date", "sale_month", "customer_id", "customer_name",
            "region", "segment", "product_id", "product_name", "category",
            "price", "quantity", "total_amount"
          )

encriched.write.mode('overwrite').saveAsTable('chat_bot.gold.sales_enriched')



monthly=encriched.groupBy('sale_month','category').agg(

    F.sum('total_amount').alias('revenue'),
    F.sum('quantity').alias('unit_sold'),
    F.countDistinct("sale_id").alias("orders"))\
    .orderBy("sale_month","category")    

monthly.write.mode('overwrite').saveAsTable('chat_bot.gold.monthly_sales_by_category')
spark.table('chat_bot.gold.monthly_sales_by_category').display()

